In [1]:
import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import pandas as pd
from sklearn import preprocessing
from sklearn.preprocessing import OneHotEncoder

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import mean_absolute_error, r2_score, median_absolute_error, mean_squared_error

from sklearn.preprocessing import MinMaxScaler

import statsmodels.api as sm

import matplotlib.pyplot as plt 
import seaborn as sns
import joblib


In [2]:
# Definada as sementes para reprodutibilidade
random_seed = 196572
np.random.seed(random_seed)

In [3]:
scaler = MinMaxScaler()
plt.rcParams["figure.figsize"] = [22,8]
le = preprocessing.LabelEncoder()

In [4]:
##############################################
# Abre o arquivo e mostra o conteúdo
df = pd.read_csv('Alunos - Dados.csv',sep=',')
#df = df.drop('num', axis = 1)
results = []
#df.info()

print(' ')
print('###################')
print('Conteúdo do arquivo - Alunos')
print(df.head())

 
###################
Conteúdo do arquivo - Alunos
  school  sex  age address famsize Pstatus  Medu  Fedu     Mjob      Fjob  \
0     GP    2   18       U     GT3       A     4     4  at_home   teacher   
1     GP    2   17       U     GT3       T     1     1  at_home     other   
2     GP    2   15       U     LE3       T     1     1  at_home     other   
3     GP    2   15       U     GT3       T     4     2   health  services   
4     GP    2   16       U     GT3       T     3     3    other     other   

   ... famrel freetime  goout  Dalc  Walc health absences  G1  G2  G3  
0  ...      4        3      4     1     1      3        6   5   6   6  
1  ...      5        3      3     1     1      3        4   5   5   6  
2  ...      4        3      2     2     3      3       10   7   8  10  
3  ...      3        2      2     1     1      5        2  15  14  15  
4  ...      4        3      2     1     2      5        4   6  10  10  

[5 rows x 33 columns]


In [5]:
##############################################
# Define as métricas
def get_regression_metrics(y_test, y_pred, modelo,params):
    metrics = {
        "MODELO":modelo,
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred),
        "MAPE": np.mean(np.abs((y_test - y_pred) / y_test)) * 100,
        "MedAE": median_absolute_error(y_test, y_pred),
        "params":params
    }
    return metrics

In [6]:
##############################################
# Discretiza as variáveis com domínio
df['school'] = le.fit_transform(df['school'])
df['address'] = le.fit_transform(df['address'])
df['famsize'] = le.fit_transform(df['famsize'])
df['Pstatus'] = le.fit_transform(df['Pstatus'])
df['Mjob'] = le.fit_transform(df['Mjob'])
df['Fjob'] = le.fit_transform(df['Fjob'])
df['reason'] = le.fit_transform(df['reason'])
df['guardian'] = le.fit_transform(df['guardian'])
df['schoolsup'] = le.fit_transform(df['schoolsup'])
df['famsup'] = le.fit_transform(df['famsup'])
df['paid'] = le.fit_transform(df['paid'])
df['activities'] = le.fit_transform(df['activities'])
df['nursery'] = le.fit_transform(df['nursery'])
df['higher'] = le.fit_transform(df['higher'])
df['internet'] = le.fit_transform(df['internet'])
df['romantic'] = le.fit_transform(df['romantic'])


In [ ]:
##############################################
# Mostra gráfico de correlação
# corr = df.corr(method='pearson')
# sns.heatmap(corr,cmap='seismic',annot=True, fmt=".2f")
# plt.show()

In [7]:
##############################################
# EXPERIMENTO - Separa as bases
y = df['G3']
#y = le.fit_transform(df['tipo'])
X = df.drop('G3', axis = 1)

columns = list(X.columns)
X = scaler.fit_transform(X)
X = pd.DataFrame(X, columns=columns)
  
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 9)

#X.head()

In [ ]:
##############################################
# EXPERIMENTO - ???????
# x = sm.add_constant(X)
# model = sm.OLS(y, x.astype(float)).fit()
# print(model.summary())
# print('R2: ', model.rsquared)


In [8]:
##############################################
# EXPERIMENTO
param_grid = {
    'n_neighbors': list(range(1, 21)),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

In [9]:
grid = GridSearchCV(KNeighborsRegressor(), param_grid, n_jobs= -1, cv=9)
grid.fit(X_train, y_train)

y_pred = grid.predict(X_test)
metrics_model = get_regression_metrics(y_test, y_pred,"KNeighborsRegressor", grid.best_estimator_)
results.append(metrics_model)

In [12]:
print(' ')
print('###########################')
print('EXPERIMENTO KNN - Alunos')
print(' ')
print('Melhores parâmetros:')
print(' ')

best_params = grid.best_estimator_
print(f"Melhores parametros: {best_params}")

mae = mean_absolute_error(y_test, y_pred)
print("Erro Médio Absoluto:", mae)

mse = mean_squared_error(y_test, y_pred)
print("Erro Quadrático Médio:", mse)

rmse = np.sqrt(mse)
print("Raiz do Erro Quadrático Médio:", rmse)

r2 = r2_score(y_test, y_pred)
print("R-quadrado:", r2)

mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
print("Erro Médio Absoluto Percentual:", mape)

medae = median_absolute_error(y_test, y_pred)
print("Erro Absoluto Mediano:", medae)

 
###########################
EXPERIMENTO KNN - Alunos
 
Melhores parâmetros:
 
Melhores parametros: KNeighborsRegressor(metric='manhattan', n_neighbors=8)
Erro Médio Absoluto: 2.172268907563025
Erro Quadrático Médio: 8.674107142857142
Raiz do Erro Quadrático Médio: 2.94518371971209
R-quadrado: 0.45975198028729003
Erro Médio Absoluto Percentual: inf
Erro Absoluto Mediano: 1.5


In [13]:
##############################################
# Predição de Novos Casos

# Salva o modelo e o scaler
joblib.dump(grid.best_estimator_, "modelo_treinado.pkl")
joblib.dump(scaler, "scaler_treinado.pkl")


['scaler_treinado.pkl']

In [14]:
# Caminhos para os arquivos
modelo_path = "modelo_treinado.pkl"
scaler_path = "scaler_treinado.pkl"
dados_novos_path = "Alunos - Novos Casos - Para Python.csv"  # CSV SEM a variável alvo

In [15]:
# Carrega o modelo e o scaler
modelo = joblib.load(modelo_path)
scaler = joblib.load(scaler_path)

In [16]:
# Lê o novo arquivo CSV sem a variável alvo
dados_novos = pd.read_csv(dados_novos_path)

In [17]:
##############################################
# Discretiza as variáveis com domínio
dados_novos['school'] = le.fit_transform(dados_novos['school'])
dados_novos['address'] = le.fit_transform(dados_novos['address'])
dados_novos['famsize'] = le.fit_transform(dados_novos['famsize'])
dados_novos['Pstatus'] = le.fit_transform(dados_novos['Pstatus'])
dados_novos['Mjob'] = le.fit_transform(dados_novos['Mjob'])
dados_novos['Fjob'] = le.fit_transform(dados_novos['Fjob'])
dados_novos['reason'] = le.fit_transform(dados_novos['reason'])
dados_novos['guardian'] = le.fit_transform(dados_novos['guardian'])
dados_novos['schoolsup'] = le.fit_transform(dados_novos['schoolsup'])
dados_novos['famsup'] = le.fit_transform(dados_novos['famsup'])
dados_novos['paid'] = le.fit_transform(dados_novos['paid'])
dados_novos['activities'] = le.fit_transform(dados_novos['activities'])
dados_novos['nursery'] = le.fit_transform(dados_novos['nursery'])
dados_novos['higher'] = le.fit_transform(dados_novos['higher'])
dados_novos['internet'] = le.fit_transform(dados_novos['internet'])
dados_novos['romantic'] = le.fit_transform(dados_novos['romantic'])

In [ ]:
# Aplica a mesma padronização dos dados
#dados_novos_scaled = scaler.transform(dados_novos)


In [18]:
# Faz a predição
#predicoes = modelo.predict(dados_novos_scaled)
predicoes = modelo.predict(dados_novos)

In [19]:
# Mostra os resultados
print("Predições:")
print(predicoes)


Predições:
[ 9.375 10.25   7.875]


In [20]:
# Salva as predições no mesmo DataFrame
dados_novos['predicao'] = predicoes

In [21]:
# Exporta para novo CSV
dados_novos.to_csv("Alunos - Novos Casos - Predicoes em Python KNN.csv", index=False)
print("\nAlunos - Novos Casos - Predicoes em Python KNN.csv'")



Alunos - Novos Casos - Predicoes em Python KNN.csv'
